In [1]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
from shapely.geometry import mapping
import numpy as np
from dataclasses import dataclass
import matplotlib.pyplot as plt
from rasterio.plot import show
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from rasterio.vrt import WarpedVRT

In [ ]:
# Define paths
habitat_path = r"C:\Users\Carlos Munoz\Documents\Ph.D\6_courses\2026_I_jsdmcapt\data\exp_1\Ceroxylon_quindiuense_mx.tif"
footprint_path = r"C:\Users\Carlos Munoz\Documents\Ph.D\6_courses\2026_I_jsdmcapt\data\colombia_hfi\IHEH_1970.tif"
shapefile_path = r"C:\Users\Carlos Munoz\Documents\Ph.D\6_courses\2026_I_jsdmcapt\data\exp_1\cocora_100km.shp"

def process_and_crop(src: rasterio.io.DatasetReader, extent_gdf: gpd.GeoDataFrame) -> tuple[np.ndarray, dict]:
    # Reproject shapefile to match the raster CRS if they differ
    if extent_gdf.crs != src.crs:
        extent_gdf = extent_gdf.to_crs(src.crs)
        
    # Extract geometry in GeoJSON format for the mask function
    geometries = [mapping(geom) for geom in extent_gdf.geometry]
    
    # Crop raster
    out_image, out_transform = mask(src, geometries, crop=True)
    
    # Update metadata to reflect the new dimensions 
    out_meta = src.meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })
    
    return out_image[0], out_meta

# Call shapefile
extent = gpd.read_file(shapefile_path)

# Open rasters, plot them, and crop
with rasterio.open(habitat_path) as h_src, rasterio.open(footprint_path) as f_src:
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    
    # Reproject extent for plotting if needed
    extent_h = extent.to_crs(h_src.crs) if extent.crs != h_src.crs else extent
    extent_f = extent.to_crs(f_src.crs) if extent.crs != f_src.crs else extent

    cmap_habitat = plt.get_cmap('Spectral_r')
    cmap_footprint = plt.get_cmap('RdYlGn_r')
    
    # Plot Habitat
    show(h_src, ax=axes[0], cmap=cmap_habitat, title='Wax Palm Habitat Suitability')
    extent_h.plot(ax=axes[0], facecolor='black', alpha=0.3, edgecolor='black', linewidth=2)
    
    # Plot Footprint
    show(f_src, ax=axes[1], cmap=cmap_footprint, title='Colombia HFP 1970')
    extent_f.plot(ax=axes[1], facecolor='black', alpha=0.3, edgecolor='black', linewidth=2)
    
    plt.tight_layout()
    plt.show()

    # Cropping
    h_layer, h_meta = process_and_crop(h_src, extent)
        
    # Use WarpedVRT to reproject f_src to match h_src's CRS and resolution before cropping
    with WarpedVRT(f_src, crs=h_src.crs, transform=h_src.transform, width=h_src.width, height=h_src.height) as vrt:
        footprint_layer, footprint_meta = process_and_crop(vrt, extent)


# Verify matrix alignment
print(f"Habitat matrix shape: {h_layer.shape}")
print(f"Footprint matrix shape: {footprint_layer.shape}")

# Verify spatial resolution
pixel_width = h_meta['transform'][0]
pixel_height = -h_meta['transform'][4]
print(f"Pixel resolution: {pixel_width} x {pixel_height}")


In [3]:
@dataclass
class SpeciesParameters:
    name: str
    K_s: float  
    G_s: float  
    S_s: float  
    D_s: float
    d_max: float    

class EnvironmentalModel:
    def __init__(self, h_layer: np.ndarray, footprint_layer: np.ndarray, params: SpeciesParameters):
        self.params = params
        self.H_c = h_layer
        self.delta_c = footprint_layer / 100.0
        self.K_c = self.H_c * self.params.K_s
        
        # Initialization: Pristine state prior to disturbance
        self.N_c = np.floor(self.K_c).astype(float)
        
    def _calculate_immigration(self) -> np.ndarray:
        # Set to zero for the isolated demographic test.
        return np.zeros_like(self.N_c)

    def step_original_captain(self):
        """
        Original CAPTAIN formulation with geometric growth and hard capacity limit.
        Equation 4: N = min(N * (1 - S*delta) * G + I, K)
        """
        surviving_pop = self.N_c * (1.0 - (self.params.S_s * self.delta_c))
        growing_pop = surviving_pop * self.params.G_s
        immigrants = self._calculate_immigration()
        
        self.N_c = np.minimum(growing_pop + immigrants, self.K_c)

    def step_logistic_captain(self):
        """
        CAPTAIN formulation with logistic growth.
        Equation 26: N = (K * Surviving * G) / (K + Surviving * (G - 1)) + I
        """
        surviving_pop = self.N_c * (1.0 - (self.params.S_s * self.delta_c))
        
        numerator = self.K_c * surviving_pop * self.params.G_s
        denominator = self.K_c + (surviving_pop * (self.params.G_s - 1.0))
        
        # Safe division to prevent errors where K_c and surviving_pop are 0
        growing_pop = np.divide(
            numerator, 
            denominator, 
            out=np.zeros_like(numerator), 
            where=denominator > 0
        )
        
        immigrants = self._calculate_immigration()
        
        self.N_c = growing_pop + immigrants

In [7]:
# 1. Calculate K_s 
# K_s based on the San Juanito pristine density (31 individuals per hectare)
calculated_K_s = 31.0 * 100 # although the area should vary a little because of the pixel size, we use 100 to maintain consistency in this first approach.

# 2. Define the biological parameters
# Dispersal components are set to 0 to isolate the local demographic equations.
palm_params = SpeciesParameters(
    name="Ceroxylum quindiuense",
    K_s= calculated_K_s,
    G_s=1.05,  # 5% intrinsic growth rate
    S_s=0.25,  # 25% mortality under maximum disturbance
    D_s=0.0,
    d_max=0
)

# 3. Initiate two independent models
model_original = EnvironmentalModel(h_layer, footprint_layer, palm_params)
model_logistic = EnvironmentalModel(h_layer, footprint_layer, palm_params)

In [ ]:
# 4. Identify specific test cells for trajectory observation
# Find a highly suitable, pristine cell (high suitability, low disturbance)
pristine_mask = (h_layer > 0.8) & (footprint_layer < 10)
pristine_coords = np.argwhere(pristine_mask)[0] 

# Find a highly suitable, highly disturbed cell 
disturbed_mask = (h_layer > 0.8) & (footprint_layer > 80)
disturbed_coords = np.argwhere(disturbed_mask)[0]

py, px = pristine_coords[0], pristine_coords[1]
dy, dx = disturbed_coords[0], disturbed_coords[1]

# 5. Run the simulation loop
time_steps = 50

traj_orig_pristine = np.zeros(time_steps)
traj_orig_disturbed = np.zeros(time_steps)
traj_log_pristine = np.zeros(time_steps)
traj_log_disturbed = np.zeros(time_steps)

for t in range(time_steps):
    # Record current abundance state
    traj_orig_pristine[t] = model_original.N_c[py, px]
    traj_orig_disturbed[t] = model_original.N_c[dy, dx]
    
    traj_log_pristine[t] = model_logistic.N_c[py, px]
    traj_log_disturbed[t] = model_logistic.N_c[dy, dx]
    
    # Advance both models
    model_original.step_original_captain()
    model_logistic.step_logistic_captain()

In [ ]:
# 6. Plot the comparative trajectories
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pristine Cell Plot
ax1.plot(traj_orig_pristine, label="Original CAPTAIN", linestyle='--', color='blue')
ax1.plot(traj_log_pristine, label="Logistic Continuous", linewidth=2, color='orange')
ax1.set_title(f"Pristine Cell Trajectory\n(Suitability = {h_layer[py, px]:.2f}, Disturbance = {model_original.delta_c[py, px]:.2f})")
ax1.set_xlabel("Time Step")
ax1.set_ylabel("Abundance")
ax1.legend()

# Disturbed Cell Plot
ax2.plot(traj_orig_disturbed, label="Original CAPTAIN", linestyle='--', color='blue')
ax2.plot(traj_log_disturbed, label="Logistic Continuous", linewidth=2, color='orange')
ax2.set_title(f"Disturbed Cell Trajectory\n(Suitability = {h_layer[dy, dx]:.2f}, Disturbance = {model_original.delta_c[dy, dx]:.2f})")
ax2.set_xlabel("Time Step")
ax2.set_ylabel("Abundance")
ax2.legend()

plt.tight_layout()
plt.show()